# IF3170 Artificial Intelligence | Tugas Besar 2

Group Number: 02

Group Members:
- Andhika Maulana Addiputra (18223005)
- Kevin Azra (18223029)
- Arqila Surya Putra (18223047)
- Muhammad Zidni Alkandi (18223071)

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import warnings
warnings.filterwarnings("ignore")

# Import other libraries
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    # accuracy_score,
    # precision_score,
    # recall_score,
    # f1_score,
    # confusion_matrix,
    # classification_report,
)

print("Libraries imported successfully!")

## Import Dataset

In [ ]:

DATA_PATH = 'https://raw.githubusercontent.com/pablonification/tubes-ai2/refs/heads/main/data/'

train_df = pd.read_csv(f"{DATA_PATH}train.csv")
test_df = pd.read_csv(f"{DATA_PATH}test.csv")

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print(f"\nTarget distribution:")
print(train_df["is_fraud"].value_counts(normalize=True))

# Exploratory Data Analysis (Optional)

Exploratory Data Analysis (EDA) is a crucial step in the data analysis process that involves examining and visualizing data sets to uncover patterns, trends, anomalies, and insights. It is the first step before applying more advanced statistical and machine learning techniques. EDA helps you to gain a deep understanding of the data you are working with, allowing you to make informed decisions and formulate hypotheses for further analysis.

In [ ]:
# Basic Info
print("="*60)
print(f"\nTrain Shape: {train_df.shape}")
print(f"Test Shape: {test_df.shape}")

print("\nINFO: Data types:")
print(train_df.dtypes.value_counts())

print("\nINFO: Unique values:")
print(train_df.nunique())

print("\nINFO: Describe:")
print(train_df.describe())

# Target Distribution
print("\n" + "="*60)
print("\nINFO: Target Distribution (is_fraud):")
target_counts = train_df['is_fraud'].value_counts()
print(f"target_counts>>>>{target_counts}")

target_pct = train_df['is_fraud'].value_counts(normalize=True) * 100
print(f"target_pct>>>>{target_pct}")

print(f"\nClass 0 (Normal): {target_counts[0]:,} ({target_pct[0]:.2f}%)")
print(f"Class 1 (Fraud):  {target_counts[1]:,} ({target_pct[1]:.2f}%)")
print(f"\nImbalance Ratio: {target_counts[0]/target_counts[1]:.2f}:1")

# Visualisasi
fig, ax = plt.subplots(1,2, figsize=(12,4))
target_counts.plot(kind='bar', ax=ax[0], color=['green', 'red'])
ax[0].set_title('Target Distribution (Count)')
ax[0].set_xlabel('is_fraud')
ax[0].set_ylabel('Count')
ax[0].set_xticklabels(['Normal (0)', 'Fraud (1)'], rotation=0)
target_pct.plot(kind='pie', ax=ax[1], autopct='%.1f%%', colors=['green', 'red'])
ax[1].set_title('Target Distribution (%)')
ax[1].set_ylabel('')
plt.tight_layout()
plt.show()

# missing values
print("\n" + "="*60)
print("\nINFO: Missing Values:")

missing = train_df.isnull().sum()
print(f"missing>>>>>{missing}")

missing_pct = (missing / len(train_df)) * 100
print(f"missing_pct>>>>>{missing_pct}")

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}).sort_values('Missing Count', ascending=False)
missing_df = missing_df[missing_df['Missing Count'] > 0]

if len(missing_df) > 0:
    print(f"\nColumns with missing values: {len(missing_df)}")
    print(missing_df)
else:
    print("\nNo missing values found!")

# Numeric Features
print("\n" + "="*60)
print("\nINFO: Numeric Features:")
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols}")
print("\nStatistics:")
train_df[numeric_cols].describe()


# 1. Split Training Set, Validation Set, Testing Set

Splitting the training and validation set works as an early diagnostic towards the performance of the model we train. This is done before the preprocessing steps to **avoid data leakage inbetween the sets**. If you want to use k-fold cross-validation, split the data later and do the cleaning and preprocessing separately for each split.

Note: For training, you should use the data contained in the `train` folder given by the TA. The `test` data is only used for kaggle submission.

In [ ]:
# Store test ID for submission
test_ids = test_df["ID"].values

# Split training data for validation in this machine
X_full = train_df.drop(columns=["is_fraud"])
y_full = train_df["is_fraud"].values

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

print(f"Training set: {X_train_raw.shape}")
print(f"Validation set: {X_val_raw.shape}")
print(f"Test set: {test_df.shape}")

# 2. Data Cleaning and Preprocessing

This step is the first thing to be done once a Data Scientist have grasped a general knowledge of the data. Raw data is **seldom ready for training**, therefore steps need to be taken to clean and format the data for the Machine Learning model to interpret.

By performing data cleaning and preprocessing, you ensure that your dataset is ready for model training, leading to more accurate and reliable machine learning results. These steps are essential for transforming raw data into a format that machine learning algorithms can effectively learn from and make predictions.

We will give some common methods for you to try, but you only have to **at least implement one method for each process**. For each step that you will do, **please explain the reason why did you do that process. Write it in a markdown cell under the code cell you wrote.**

## A. Data Cleaning

**Data cleaning** is the crucial first step in preparing your dataset for machine learning. Raw data collected from various sources is often messy and may contain errors, missing values, and inconsistencies. Data cleaning involves the following steps:

1. **Handling Missing Data:** Identify and address missing values in the dataset. This can include imputing missing values, removing rows or columns with excessive missing data, or using more advanced techniques like interpolation.

2. **Dealing with Outliers:** Identify and handle outliers, which are data points significantly different from the rest of the dataset. Outliers can be removed or transformed to improve model performance.

3. **Data Validation:** Check for data integrity and consistency. Ensure that data types are correct, categorical variables have consistent labels, and numerical values fall within expected ranges.

4. **Removing Duplicates:** Identify and remove duplicate rows, as they can skew the model's training process and evaluation metrics.

5. **Feature Engineering**: Create new features or modify existing ones to extract relevant information. This step can involve scaling, normalizing, or encoding features for better model interpretability.

### I. Handling Missing Data

Missing data can adversely affect the performance and accuracy of machine learning models. There are several strategies to handle missing data in machine learning:

1. **Data Imputation:**

    a. **Mean, Median, or Mode Imputation:** For numerical features, you can replace missing values with the mean, median, or mode of the non-missing values in the same feature. This method is simple and often effective when data is missing at random.

    b. **Constant Value Imputation:** You can replace missing values with a predefined constant value (e.g., 0) if it makes sense for your dataset and problem.

    c. **Imputation Using Predictive Models:** More advanced techniques involve using predictive models to estimate missing values. For example, you can train a regression model to predict missing numerical values or a classification model to predict missing categorical values.

2. **Deletion of Missing Data:**

    a. **Listwise Deletion:** In cases where the amount of missing data is relatively small, you can simply remove rows with missing values from your dataset. However, this approach can lead to a loss of valuable information.

    b. **Column (Feature) Deletion:** If a feature has a large number of missing values and is not critical for your analysis, you can consider removing that feature altogether.

3. **Domain-Specific Strategies:**

    a. **Domain Knowledge:** In some cases, domain knowledge can guide the imputation process. For example, if you know that missing values are related to a specific condition, you can impute them accordingly.

4. **Imputation Libraries:**

    a. **Scikit-Learn:** Scikit-Learn provides a `SimpleImputer` class that can handle basic imputation strategies like mean, median, and mode imputation.

    b. **Fancyimpute:** Fancyimpute is a Python library that offers more advanced imputation techniques, including matrix factorization, k-nearest neighbors, and deep learning-based methods.

The choice of imputation method should be guided by the nature of your data, the amount of missing data, the problem you are trying to solve, and the assumptions you are willing to make.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureImputer(BaseEstimator, TransformerMixin):
    """
    Transformer dari sklearn ini digunakan untuk menangani nilai yang hilang dalam data.
    - Numeric columns: fill with median -> karna ada outlier dan untuk preserve distribusi datanya
    - Categorical columns: fill with mode -> memperlihatakan seberapa banyak
    """
    
    def __init__(self):
        self.medians_ = {}
        self.modes_ = {}
        self.numeric_cols_ = []
        self.categorical_cols_ = []
    
    def fit(self, X, y=None):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Identify column types
        self.numeric_cols_ = df.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols_ = df.select_dtypes(include=["object"]).columns.tolist()
        
        # Exclude ID columns from imputation
        id_cols = ["ID", "transaction_id", "user_id"]
        self.numeric_cols_ = [c for c in self.numeric_cols_ if c not in id_cols]
        
        # Compute medians for numeric columns
        for col in self.numeric_cols_:
            self.medians_[col] = df[col].median()
        
        # Compute modes for categorical columns
        for col in self.categorical_cols_:
            self.modes_[col] = df[col].mode()[0] if len(df[col].mode()) > 0 else "Unknown"
        
        return self
    
    def transform(self, X):
        df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Fill numeric columns with median
        for col in self.numeric_cols_:
            if col in df.columns:
                df[col] = df[col].fillna(self.medians_.get(col, 0))
        
        # Fill categorical columns with mode
        for col in self.categorical_cols_:
            if col in df.columns:
                df[col] = df[col].fillna(self.modes_.get(col, "Unknown"))
        
        return df


# Apply imputation
imputer = FeatureImputer()
train_df_clean = imputer.fit_transform(train_df)
test_df_clean = imputer.transform(test_df)

print("Missing values after cleaning:")
print(f"Train: {train_df_clean.isnull().sum().sum()}")
print(f"Test: {test_df_clean.isnull().sum().sum()}")

### II. Dealing with Outliers

Outliers are data points that significantly differ from the majority of the data. They can be unusually high or low values that do not fit the pattern of the rest of the dataset. Outliers can significantly impact model performance, so it is important to handle them properly.

Some methods to handle outliers:
1. **Imputation**: Replace with mean, median, or a boundary value.
2. **Clipping**: Cap values to upper and lower limits.
3. **Transformation**: Use log, square root, or power transformations to reduce their influence.
4. **Model-Based**: Use algorithms robust to outliers (e.g., tree-based models, Huber regression).

In [ ]:
class OutlierHandler(BaseEstimator, TransformerMixin):
    """
    Menggunakan transformer yang sama untuk handling outliers via CLIPPING -> mengganti outlier dengan batas yg ditentukan instead of ngapus
    Pake IQR = Q3 - Q1
    """
    
    def __init__(self, clip_cols=None, clip_bounds=None):
        # Specific columns and their clip bounds
        self.clip_cols = clip_cols or []
        self.clip_bounds = clip_bounds or {}
        self.computed_bounds_ = {}
    
    def fit(self, X, y=None):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # For columns without predefined bounds, compute IQR-based bounds
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        id_cols = ["ID", "transaction_id", "user_id", "is_fraud"]
        numeric_cols = [c for c in numeric_cols if c not in id_cols]
        
        for col in numeric_cols:
            if col not in self.clip_bounds:
                Q1 = df[col].quantile(0.01)
                Q3 = df[col].quantile(0.99)
                self.computed_bounds_[col] = (Q1, Q3)
        
        return self
    
    def transform(self, X):
        df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Apply predefined bounds
        for col, (lower, upper) in self.clip_bounds.items():
            if col in df.columns:
                df[col] = df[col].clip(lower, upper)
        # IQR baru dilakukan pada feature engineering agar original behavior tetap terjaga
        
        return df


# Define explicit outlier handling bounds
# These are applied during feature engineering to specific derived features
outlier_bounds = {
    "amount_zscore": (-10, 10),
    "amount_ratio": (0, 100),
}

outlier_handler = OutlierHandler(clip_bounds=outlier_bounds)
print("Outlier handling configured:")
print(f"  - amount_zscore: clipped to [-10, 10]")
print(f"  - amount_ratio: clipped to [0, 100]")
print("Outliers in derived features will be clipped during feature engineering.")

### III. Remove Duplicates
Handling duplicate values is crucial because they can compromise data integrity, leading to inaccurate analysis and insights. Duplicate entries can bias machine learning models, causing overfitting and reducing their ability to generalize to new data. They also inflate the dataset size unnecessarily, increasing computational costs and processing times. Additionally, duplicates can distort statistical measures and lead to inconsistencies, ultimately affecting the reliability of data-driven decisions and reporting. Ensuring data quality by removing duplicates is essential for accurate, efficient, and consistent analysis.

In [ ]:
print(f"Duplicates in train: {train_df_clean.duplicated().sum()}")
print(f"Duplicates in test: {test_df_clean.duplicated().sum()}")
# gaperlu karna gaada

### IV. Feature Engineering

**Feature engineering** involves creating new features (input variables) or transforming existing ones to improve the performance of machine learning models. Feature engineering aims to enhance the model's ability to learn patterns and make accurate predictions from the data. It's often said that "good features make good models."

1. **Feature Selection:** Feature engineering can involve selecting the most relevant and informative features from the dataset. Removing irrelevant or redundant features not only simplifies the model but also reduces the risk of overfitting.

2. **Creating New Features:** Sometimes, the existing features may not capture the underlying patterns effectively. In such cases, engineers create new features that provide additional information. For example:
   
   - **Polynomial Features:** Engineers may create new features by taking the square, cube, or other higher-order terms of existing numerical features. This can help capture nonlinear relationships.
   
   - **Interaction Features:** Interaction features are created by combining two or more existing features. For example, if you have features "length" and "width," you can create an "area" feature by multiplying them.

3. **Binning or Discretization:** Continuous numerical features can be divided into bins or categories. For instance, age values can be grouped into bins like "child," "adult," and "senior."

4. **Domain-Specific Feature Engineering:** Depending on the domain and problem, engineers may create domain-specific features. For example, in fraud detection, features related to transaction history and user behavior may be engineered to identify anomalies.

Feature engineering is both a creative and iterative process. It requires a deep understanding of the data, domain knowledge, and experimentation to determine which features will enhance the model's predictive power.

In [ ]:
class FeatureCreator(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.quantiles_ = {}
    
    def fit(self, X, y=None):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Compute quantiles from training data only (prevent data leakage)
        self.quantiles_ = {
            "distance_q75": df["distance_from_home"].quantile(0.75),
            "amount_q90": df["transaction_amount"].quantile(0.90),
        }
        
        return self
    
    def transform(self, X):
        df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Amount features
        std_val = df["std_transaction_amount"].replace(0, 1)
        df["amount_zscore"] = (
            (df["transaction_amount"] - df["avg_transaction_amount"]) / std_val
        ).clip(-10, 10)  # Outlier clipping
        df["amount_ratio"] = (
            df["transaction_amount"] / (df["avg_transaction_amount"] + 1)
        ).clip(0, 100)  # Outlier clipping

        # Velocity features
        df["high_velocity_2"] = (df["transactions_last_1h"] >= 2).astype(int) # astype is assigning datatype to int
        df["high_velocity_3"] = (df["transactions_last_1h"] >= 3).astype(int)

        # Risk features
        df["risk_sum"] = df["ip_risk_score"] + df["merchant_risk"] + df["country_risk"]
        df["risk_weighted"] = df["ip_risk_score"] * (1 - df["device_trust_score"] + 0.01)
        df["final_risk_score_v2"] = (
            df["ip_risk_score"] * 0.4
            + df["merchant_risk"] * 0.3
            + (1 - df["device_trust_score"]) * 0.3
        )

        # Location/account features
        df["far_from_home"] = (df["distance_from_home"] > self.quantiles_["distance_q75"]).astype(int)
        df["account_new"] = (df["account_age_days"] < 30).astype(int)

        # Log features
        df["log_transaction_amount"] = np.log1p(df["transaction_amount"]) # log1p(x) = hitung log(1+x); lebih aman dari log(x) karena x bisa 0 -> log(0) = -inf
        df["log_distance"] = np.log1p(df["distance_from_home"])

        # Interaction features
        df["risk_x_amount"] = df["ip_risk_score"] * df["amount_ratio"]
        df["velocity_x_risk"] = df["transactions_last_1h"] * df["ip_risk_score"]
        df["age_inv_x_risk"] = df["ip_risk_score"] / (df["account_age_days"] + 1)
        df["distance_x_newcountry"] = df["distance_from_home"] * df["is_new_country"]
        df["trust_deficit"] = (1 - df["device_trust_score"]) * df["ip_risk_score"]
        df["shared_resource_score"] = df["shared_ip_users"] + df["shared_device_users"]

        # Tier features
        df["high_amount"] = (df["transaction_amount"] > self.quantiles_["amount_q90"]).astype(int)
        df["risk_tier_high"] = (df["risk_sum"] > 1.5).astype(int)
        df["risk_tier_extreme"] = (df["risk_sum"] > 2.0).astype(int)

        return df


# Apply feature engineering
feature_creator = FeatureCreator()
train_fe = feature_creator.fit_transform(train_df_clean)
test_fe = feature_creator.transform(test_df_clean)

print(f"Features after engineering: {train_fe.shape[1]}")
print(f"New features created: 19")

## B. Data Preprocessing

**Data preprocessing** is a broader step that encompasses both data cleaning and additional transformations to make the data suitable for machine learning algorithms. Its primary goals are:

1. **Feature Scaling:** Ensure that numerical features have similar scales. Common techniques include Min-Max scaling (scaling to a specific range) or standardization (mean-centered, unit variance).

2. **Encoding Categorical Variables:** Machine learning models typically work with numerical data, so categorical variables need to be encoded. This can be done using one-hot encoding, label encoding, or more advanced methods like target encoding.

3. **Handling Imbalanced Classes:** If dealing with imbalanced classes in a binary classification task, apply techniques such as oversampling, undersampling, or using different evaluation metrics to address class imbalance.

4. **Dimensionality Reduction:** Reduce the number of features using techniques like Principal Component Analysis (PCA) or feature selection to simplify the model and potentially improve its performance.

5. **Normalization:** Normalize data to achieve a standard distribution. This is particularly important for algorithms that assume normally distributed data.

### Notes on Preprocessing processes

It is advised to create functions or classes that have the same/similar type of inputs and outputs, so you can add, remove, or swap the order of the processes easily. You can implement the functions or classes by yourself

or

use `sklearn` library. To create a new preprocessing component in `sklearn`, implement a corresponding class that includes:
1. Inheritance to `BaseEstimator` and `TransformerMixin`
2. The method `fit`
3. The method `transform`

### I. Feature Scaling & II. Feature Encoding

**Feature scaling** is a preprocessing technique used in machine learning to standardize the range of independent variables or features of data. The primary goal of feature scaling is to ensure that all features contribute equally to the training process and that machine learning algorithms can work effectively with the data.

Here are the main reasons why feature scaling is important:

1. **Algorithm Sensitivity:** Many machine learning algorithms are sensitive to the scale of input features. If the scales of features are significantly different, some algorithms may perform poorly or take much longer to converge.

2. **Distance-Based Algorithms:** Algorithms that rely on distances or similarities between data points, such as k-nearest neighbors (KNN) and support vector machines (SVM), can be influenced by feature scales. Features with larger scales may dominate the distance calculations.

3. **Regularization:** Regularization techniques, like L1 (Lasso) and L2 (Ridge) regularization, add penalty terms based on feature coefficients. Scaling ensures that all features are treated equally in the regularization process.

Common methods for feature scaling include:

1. **Min-Max Scaling (Normalization):** This method scales features to a specific range, typically [0, 1]. It's done using the following formula:

   $$X' = \frac{X - X_{min}}{X_{max} - X_{min}}$$

   - Here, $X$ is the original feature value, $X_{min}$ is the minimum value of the feature, and $X_{max}$ is the maximum value of the feature.  
<br />
<br />
2. **Standardization (Z-score Scaling):** This method scales features to have a mean (average) of 0 and a standard deviation of 1. It's done using the following formula:

   $$X' = \frac{X - \mu}{\sigma}$$

   - $X$ is the original feature value, $\mu$ is the mean of the feature, and $\sigma$ is the standard deviation of the feature.  
<br />
<br />
3. **Robust Scaling:** Robust scaling is a method that scales features to the interquartile range (IQR) and is less affected by outliers. It's calculated as:

   $$X' = \frac{X - Q1}{Q3 - Q1}$$

   - $X$ is the original feature value, $Q1$ is the first quartile (25th percentile), and $Q3$ is the third quartile (75th percentile) of the feature.  
<br />
<br />
4. **Log Transformation:** In cases where data is highly skewed or has a heavy-tailed distribution, taking the logarithm of the feature values can help stabilize the variance and improve scaling.

The choice of scaling method depends on the characteristics of your data and the requirements of your machine learning algorithm. **Min-max scaling and standardization are the most commonly used techniques and work well for many datasets.**

Scaling should be applied separately to each training and test set to prevent data leakage from the test set into the training set. Additionally, **some algorithms may not require feature scaling, particularly tree-based models.**

---

**Feature encoding**, also known as **categorical encoding**, is the process of converting categorical data (non-numeric data) into a numerical format so that it can be used as input for machine learning algorithms. Most machine learning models require numerical data for training and prediction, so feature encoding is a critical step in data preprocessing.

Categorical data can take various forms, including:

1. **Nominal Data:** Categories with no intrinsic order, like colors or country names.  

2. **Ordinal Data:** Categories with a meaningful order but not necessarily equidistant, like education levels (e.g., "high school," "bachelor's," "master's").

There are several common methods for encoding categorical data:

1. **Label Encoding:**

   - Label encoding assigns a unique integer to each category in a feature.
   - It's suitable for ordinal data where there's a clear order among categories.
   - For example, if you have an "education" feature with values "high school," "bachelor's," and "master's," you can encode them as 0, 1, and 2, respectively.
<br />
<br />
2. **One-Hot Encoding:**

   - One-hot encoding creates a binary (0 or 1) column for each category in a nominal feature.
   - It's suitable for nominal data where there's no inherent order among categories.
   - Each category becomes a new feature, and the presence (1) or absence (0) of a category is indicated for each row.
<br />
<br />
3. **Target Encoding (Mean Encoding):**

   - Target encoding replaces each category with the mean of the target variable for that category.
   - It's often used for classification problems.

In [ ]:
class FeatureEncoder(BaseEstimator, TransformerMixin):
    """
    Sklearn-compatible transformer for one-hot encoding categorical features.
    """

    def __init__(self):
        self.encoded_columns_ = None
        self.cat_cols_ = []
        self.id_cols_to_drop = ["ID", "transaction_id", "user_id"]

    def fit(self, X, y=None):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        
        # Drop ID columns before identifying categorical columns for encoding
        df_processed = df.drop(columns=[col for col in self.id_cols_to_drop if col in df.columns], errors='ignore')

        # Identify categorical columns
        self.cat_cols_ = df_processed.select_dtypes(include=["object"]).columns.tolist()

        # Perform one-hot encoding to get column names
        df_encoded = pd.get_dummies(df_processed, columns=self.cat_cols_, drop_first=False)
        self.encoded_columns_ = df_encoded.columns.tolist()

        return self

    def transform(self, X):
        df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)

        # Drop ID columns in transform as well
        df_processed = df.drop(columns=[col for col in self.id_cols_to_drop if col in df.columns], errors='ignore')

        # One-hot encode
        df = pd.get_dummies(df_processed, columns=self.cat_cols_, drop_first=False)

        # Align columns with training data
        for col in self.encoded_columns_:
            if col not in df.columns:
                df[col] = 0

        # Keep only columns from training
        df = df[[c for c in self.encoded_columns_ if c in df.columns]]

        return df


class FeatureSelector(BaseEstimator, TransformerMixin):
    """
    Sklearn-compatible transformer for selecting specific features.
    """

    def __init__(self, features):
        self.features = features
        self.available_features_ = []

    def fit(self, X, y=None):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.available_features_ = [f for f in self.features if f in df.columns]
        return self

    def transform(self, X):
        df = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)

        # Select available features
        result = df[self.available_features_].copy()

        # Convert to numpy and handle NaN/Inf
        arr = result.values.astype(np.float64)
        arr = np.nan_to_num(arr, nan=0, posinf=0, neginf=0)

        return arr


class FeatureScaler(BaseEstimator, TransformerMixin):
    """
    Sklearn-compatible transformer for StandardScaler.
    """

    def __init__(self):
        self.scaler_ = StandardScaler()

    def fit(self, X, y=None):
        self.scaler_.fit(X)
        return self

    def transform(self, X):
        return self.scaler_.transform(X)


# Define feature sets
FEATURES_27 = [  # For Decision Tree
    "transaction_amount",
    "amount_zscore",
    "avg_transaction_amount",
    "std_transaction_amount",
    "amount_ratio",
    "transactions_last_24h",
    "transactions_last_1h",
    "high_velocity_2",
    "high_velocity_3",
    "ip_risk_score",
    "device_trust_score",
    "merchant_risk",
    "risk_sum",
    "risk_weighted",
    "final_risk_score_v2",
    "distance_from_home",
    "far_from_home",
    "is_new_country",
    "account_age_days",
    "account_new",
    "has_chargeback_history",
    "shared_ip_users",
    "shared_device_users",
    "failed_login_attempts",
    "device_type_desktop",
    "device_type_mobile",
    "device_type_tablet",
]

FEATURES_38 = FEATURES_27 + [  # For Logistic Regression
    "log_transaction_amount",
    "log_distance",
    "risk_x_amount",
    "velocity_x_risk",
    "age_inv_x_risk",
    "distance_x_newcountry",
    "trust_deficit",
    "shared_resource_score",
    "high_amount",
    "risk_tier_high",
    "risk_tier_extreme",
]

# Apply encoding
encoder = FeatureEncoder()
train_encoded = encoder.fit_transform(train_fe)
test_encoded = encoder.transform(test_fe)

# Extract target
y = train_df["is_fraud"].values

# Select features for DT (27 features)
selector_dt = FeatureSelector(FEATURES_27)
X_27 = selector_dt.fit_transform(train_encoded)
X_test_27 = selector_dt.transform(test_encoded)

# Select features for LR (38 features)
selector_lr = FeatureSelector(FEATURES_38)
X_38 = selector_lr.fit_transform(train_encoded)
X_test_38 = selector_lr.transform(test_encoded)

print(f"DT features: {X_27.shape[1]} (target: 27)")
print(f"LR features: {X_38.shape[1]} (target: 38)")

# Scale LR features
scaler = FeatureScaler()
X_38_scaled = scaler.fit_transform(X_38)
X_test_38_scaled = scaler.transform(X_test_38)

print("\nData preparation complete!")

### III. Handling Imbalanced Dataset

**Handling imbalanced datasets** is important because imbalanced data can lead to several issues that negatively impact the performance and reliability of machine learning models. Here are some key reasons:

1. **Biased Model Performance**:

 - Models trained on imbalanced data tend to be biased towards the majority class, leading to poor performance on the minority class. This can result in misleading accuracy metrics.

2. **Misleading Accuracy**:

 - High overall accuracy can be misleading in imbalanced datasets. For example, if 95% of the data belongs to one class, a model that always predicts the majority class will have 95% accuracy but will fail to identify the minority class.

3. **Poor Generalization**:

 - Models trained on imbalanced data may not generalize well to new, unseen data, especially if the minority class is underrepresented.


Some methods to handle imbalanced datasets:
1. **Resampling Methods**:

 - Oversampling: Increase the number of instances in the minority class by duplicating or generating synthetic samples (e.g., SMOTE).
 - Undersampling: Reduce the number of instances in the majority class to balance the dataset.

2. **Evaluation Metrics**:

 - Use appropriate evaluation metrics such as precision, recall, F1-score, ROC-AUC, and confusion matrix instead of accuracy to better assess model performance on imbalanced data.

3. **Algorithmic Approaches**:

 - Use algorithms that are designed to handle imbalanced data, such as decision trees, random forests, or ensemble methods.
 - Adjust class weights in algorithms to give more importance to the minority class.

In [ ]:
# Class distribution analysis
class_counts = np.bincount(y)
class_ratio = class_counts[0] / class_counts[1]

print(f"Class distribution:")
print(f"  Class 0 (Normal): {class_counts[0]:,} ({class_counts[0]/len(y)*100:.2f}%)")
print(f"  Class 1 (Fraud):  {class_counts[1]:,} ({class_counts[1]/len(y)*100:.2f}%)")
print(f"  Imbalance ratio: {class_ratio:.2f}:1")

print("\n" + "="*60)
print("HANDLING IMBALANCED DATA: PROBABILITY CALIBRATION")
print("="*60)
"""
Disini dipakai PROBABILITY CALIBRATION instead of resampling karna:
1. Menjaga distribusi data asli - tidak ada synthetic samples
2. Meningkatkan estimasi probabilitas untuk kelas minoritas (fraud)
3. Cocok dengan metrik AUC-ROC yang digunakan dalam Kaggle
4. Mengurangi prediksi yang overconfident pada kelas mayoritas

Methods Used:
- Logistic Regression: Sigmoid (Platt) Calibration with CV=4
- Decision Tree: Isotonic Calibration with CV=5

Keduanya di implementasi dari scratch dengan:
- SigmoidCalibrationScratch: L-BFGS optimization for Platt scaling
- IsotonicRegressionScratch: Pool Adjacent Violators Algorithm (PAVA)
- CalibratedClassifierCVScratch: Cross-validation wrapper
"""

### IV. Data Normalization

Data normalization is used to achieve a standard distribution. Without normalization, models or processes that rely on the assumption of normality may not work correctly. Normalization helps reduce the magnitude effect and ensures numerical stability during optimization.

In [ ]:
# dipakai untuk logreg dan dilakukan Scaling dengan StandardScaler; udah dilakukan di feature scaling
print("Already done in feature scaling")

### V. Dimensionality Reduction

Dimensionality reduction is a technique used in data preprocessing to reduce the number of input features (dimensions) in a dataset while retaining as much important information as possible. It is essential when dealing with high-dimensional data, where too many features can cause problems like increased computational costs, overfitting, and difficulty in visualization. Reducing dimensions simplifies the data, making it easier to analyze and improving the performance of machine learning models.

One of the main approaches to dimensionality reduction is feature extraction. Feature extraction creates new, smaller sets of features that capture the essence of the original data. Common techniques include:

1. **Principal Component Analysis (PCA)**: Converts correlated features into a smaller number of uncorrelated "principal components."
2. **t-SNE (t-Distributed Stochastic Neighbor Embedding)**: A visualization-focused method to project high-dimensional data into 2D or 3D spaces.
3. **Autoencoders**: Neural networks that learn compressed representations of the data.

In [ ]:
# Write your code here

# 3. Compile Preprocessing Pipeline

All of the preprocessing classes or functions defined earlier will be compiled in this step.

If you use sklearn to create preprocessing classes, you can list your preprocessing classes in the Pipeline object sequentially, and then fit and transform your data.

In [ ]:
# from sklearn.pipeline import Pipeline

# # Note: You can add or delete preprocessing components from this pipeline

# pipe = Pipeline([("imputer", FeatureImputer()),
#                  ("featurecreator", FeatureCreator()),
#                  ("scaler", FeatureScaler()),
#                  ("encoder", FeatureEncoder())])

# train_set = pipe.fit_transform(train_set)
# val_set = pipe.transform(val_set)

# Pipeline for Decision Tree (no scaling needed)
from sklearn.pipeline import Pipeline

pipeline_dt = Pipeline([
    ("imputer", FeatureImputer()),
    ("feature_creator", FeatureCreator()),
    ("encoder", FeatureEncoder()),
    ("selector", FeatureSelector(FEATURES_27)),
])

# Pipeline for Logistic Regression (with scaling)
pipeline_lr = Pipeline([
    ("imputer", FeatureImputer()),
    ("feature_creator", FeatureCreator()),
    ("encoder", FeatureEncoder()),
    ("selector", FeatureSelector(FEATURES_38)),
    ("scaler", FeatureScaler()),
])

# Fit and transform using pipelines
X_dt = pipeline_dt.fit_transform(train_df)
X_test_dt = pipeline_dt.transform(test_df)

X_lr = pipeline_lr.fit_transform(train_df)
X_test_lr = pipeline_lr.transform(test_df)

print("Preprocessing pipelines compiled and applied!")
print(f"Pipeline DT output shape: {X_dt.shape}")
print(f"Pipeline LR output shape: {X_lr.shape}")

# Verify consistency with manual preprocessing
print(f"\nVerification:")
print(f"  DT features match: {np.allclose(X_dt, X_27)}")
print(f"  LR features match: {np.allclose(X_lr, X_38_scaled)}")

In [ ]:
# # Your code should work up until this point
# train_set = pipe.fit_transform(train_set)
# val_set = pipe.transform(val_set)

# 4. Modeling and Validation

Modelling is the process of building your own machine learning models to solve specific problems, or in this assignment context, predicting the target feature `attack_cat`. Validation is the process of evaluating your trained model using the validation set or cross-validation method and providing some metrics that can help you decide what to do in the next iteration of development.

## A. DTL

In [ ]:
# Type your code here

## B. Logistic Regression

In [ ]:
class IsotonicRegressionScratch:
    """
    Isotonic Regression menggunakan Pool Adjacent Violators Algorithm (PAVA).
    Digunakan untuk probability calibration.
    """

    def __init__(self, out_of_bounds="clip"):
        self.out_of_bounds = out_of_bounds
        self.X_thresholds_ = None
        self.y_thresholds_ = None
        self.X_min_, self.X_max_ = None, None
        self.y_min_, self.y_max_ = None, None

    def fit(self, X, y, sample_weight=None):
        """Fit isotonic regression using PAVA algorithm."""
        X = np.asarray(X, dtype=np.float64).ravel()
        y = np.asarray(y, dtype=np.float64).ravel()

        if sample_weight is None:
            sample_weight = np.ones(len(X))

        # Sort by X
        order = np.argsort(X)
        X, y = X[order], y[order]
        sample_weight = sample_weight[order]

        # PAVA algorithm
        n = len(y)
        weighted_y = y * sample_weight

        blocks = [
            {
                "start": i,
                "end": i + 1,
                "sum_wy": weighted_y[i],
                "sum_w": sample_weight[i],
                "value": y[i],
            }
            for i in range(n)
        ]

        i = 0
        while i < len(blocks) - 1:
            if blocks[i]["value"] > blocks[i + 1]["value"]:
                # Merge blocks
                blocks[i]["end"] = blocks[i + 1]["end"]
                blocks[i]["sum_wy"] += blocks[i + 1]["sum_wy"]
                blocks[i]["sum_w"] += blocks[i + 1]["sum_w"]
                blocks[i]["value"] = blocks[i]["sum_wy"] / blocks[i]["sum_w"]
                blocks.pop(i + 1)
                if i > 0:
                    i -= 1
            else:
                i += 1

        # Build thresholds
        self.X_thresholds_, self.y_thresholds_ = [], []
        for block in blocks:
            for x_val in X[block["start"] : block["end"]]:
                self.X_thresholds_.append(x_val)
                self.y_thresholds_.append(block["value"])

        self.X_thresholds_ = np.array(self.X_thresholds_)
        self.y_thresholds_ = np.array(self.y_thresholds_)

        # Remove duplicates
        unique_mask = np.concatenate([[True], np.diff(self.X_thresholds_) > 0])
        self.X_thresholds_ = self.X_thresholds_[unique_mask]
        self.y_thresholds_ = self.y_thresholds_[unique_mask]

        self.X_min_, self.X_max_ = self.X_thresholds_[0], self.X_thresholds_[-1]
        self.y_min_, self.y_max_ = self.y_thresholds_[0], self.y_thresholds_[-1]

        return self

    def predict(self, X):
        """Predict using linear interpolation."""
        X = np.asarray(X, dtype=np.float64).ravel()
        y_pred = np.interp(X, self.X_thresholds_, self.y_thresholds_)

        if self.out_of_bounds == "clip":
            y_pred = np.clip(y_pred, self.y_min_, self.y_max_)

        return np.clip(y_pred, 0.0, 1.0)


class SigmoidCalibrationScratch:
    """
    Sigmoid (Platt) Kalibrasi menggunakan L-BFGS optimization yg dibuat dari scratch.
    Cocok untuk: P(y=1|f) = 1 / (1 + exp(-(A*f + B)))
    """

    def __init__(self, max_iter=1000, tol=1e-7):
        self.max_iter = max_iter
        self.tol = tol
        self.A_, self.B_ = None, None

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y, sample_weight=None):
        # Fit sigmoid calibration using L-BFGS.
        X = np.asarray(X, dtype=np.float64).ravel()
        y = np.asarray(y, dtype=np.float64).ravel()
        n = len(y)

        if sample_weight is None:
            sample_weight = np.ones(n)

        # Platt's targets
        n_pos, n_neg = np.sum(y == 1), n - np.sum(y == 1)
        t_pos = (n_pos + 1.0) / (n_pos + 2.0)
        t_neg = 1.0 / (n_neg + 2.0)
        T = np.where(y == 1, t_pos, t_neg)

        # Initialize
        A = 0.0
        B = np.log((n_neg + 1.0) / (n_pos + 1.0))

        def objective_and_gradient(params):
            A_val, B_val = params
            p = self._sigmoid(A_val * X + B_val)
            p = np.clip(p, 1e-15, 1 - 1e-15)
            loss = -np.sum(sample_weight * (T * np.log(p) + (1 - T) * np.log(1 - p)))
            error = p - T
            dA = np.sum(sample_weight * error * X)
            dB = np.sum(sample_weight * error)
            return loss, np.array([dA, dB])

        def line_search(params, direction, grad, loss, max_iter=20):
            c1, alpha, rho = 1e-4, 1.0, 0.5
            directional_deriv = np.dot(grad, direction)
            for _ in range(max_iter):
                new_params = params + alpha * direction
                new_loss, _ = objective_and_gradient(new_params)
                if new_loss <= loss + c1 * alpha * directional_deriv:
                    return alpha, new_params, new_loss
                alpha *= rho
            return (
                alpha,
                params + alpha * direction,
                objective_and_gradient(params + alpha * direction)[0],
            )

        def lbfgs_two_loop(grad, s_hist, y_hist, rho_hist):
            q = grad.copy()
            m_curr = len(s_hist)
            if m_curr == 0:
                return -grad
            alpha_arr = np.zeros(m_curr)
            for i in range(m_curr - 1, -1, -1):
                alpha_arr[i] = rho_hist[i] * np.dot(s_hist[i], q)
                q = q - alpha_arr[i] * y_hist[i]
            gamma = np.dot(s_hist[-1], y_hist[-1]) / (
                np.dot(y_hist[-1], y_hist[-1]) + 1e-10
            )
            r = gamma * q
            for i in range(m_curr):
                beta = rho_hist[i] * np.dot(y_hist[i], r)
                r = r + (alpha_arr[i] - beta) * s_hist[i]
            return -r

        # L-BFGS optimization
        params = np.array([A, B])
        m = 10
        s_history, y_history, rho_history = [], [], []
        loss, grad = objective_and_gradient(params)

        for _ in range(self.max_iter):
            direction = lbfgs_two_loop(grad, s_history, y_history, rho_history)
            alpha, new_params, new_loss = line_search(params, direction, grad, loss)
            _, new_grad = objective_and_gradient(new_params)

            s, y_vec = new_params - params, new_grad - grad
            if np.linalg.norm(new_grad) < self.tol:
                params = new_params
                break

            sy = np.dot(s, y_vec)
            if sy > 1e-10:
                if len(s_history) >= m:
                    s_history.pop(0)
                    y_history.pop(0)
                    rho_history.pop(0)
                s_history.append(s)
                y_history.append(y_vec)
                rho_history.append(1.0 / sy)

            params, loss, grad = new_params, new_loss, new_grad

        self.A_, self.B_ = params[0], params[1]
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64).ravel()
        return np.clip(self._sigmoid(self.A_ * X + self.B_), 0.0, 1.0)


class CalibratedClassifierCVScratch:
    """
    Probability calibration dengan cross-validation yg dibuat dari scratch.
    Implementasi sama seperti sklearn's CalibratedClassifierCV.
    """

    def __init__(self, base_estimator, method="sigmoid", cv=5, random_state=None):
        self.base_estimator = base_estimator
        self.method = method
        self.cv = cv
        self.random_state = random_state
        self.calibrated_classifiers_ = []
        self.classes_ = None

    def _get_calibrator(self):
        if self.method == "sigmoid":
            return SigmoidCalibrationScratch(max_iter=1000, tol=1e-7)
        elif self.method == "isotonic":
            return IsotonicRegressionScratch(out_of_bounds="clip")
        else:
            raise ValueError(f"Unknown method: {self.method}")

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)

        self.classes_ = np.unique(y)
        self.calibrated_classifiers_ = []

        cv = StratifiedKFold(
            n_splits=self.cv, shuffle=True, random_state=self.random_state
        )

        for train_idx, val_idx in cv.split(X, y):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model = deepcopy(self.base_estimator)
            model.fit(X_train, y_train)

            val_proba = model.predict_proba(X_val)[:, 1]

            calibrator = self._get_calibrator()
            calibrator.fit(val_proba, y_val)

            self.calibrated_classifiers_.append((model, calibrator))

        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float64)
        all_proba = []

        for model, calibrator in self.calibrated_classifiers_:
            raw_proba = model.predict_proba(X)[:, 1]
            calibrated_proba = calibrator.predict(raw_proba)
            all_proba.append(calibrated_proba)

        mean_proba_1 = np.mean(all_proba, axis=0)
        return np.column_stack([1 - mean_proba_1, mean_proba_1])

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]


print("All calibration classes defined!")

In [ ]:
class LogisticRegressionScratch:
    """
    Logistic Regression implemented from scratch pake L-BFGS optimization.

    Features:
    - L-BFGS optimizer with two-loop recursion
    - L2 regularization (scaled by n_samples to match sklearn)
    - Numerically stable sigmoid
    """

    def __init__(
        self,
        max_iter=200,
        tol=1e-6,
        C=1.0,
        regularization="l2",
        random_state=None,
        verbose=False,
    ):
        self.max_iter = max_iter
        self.tol = tol
        self.C = C
        self.regularization = regularization
        self.random_state = random_state
        self.verbose = verbose
        self.weights_ = None
        self.bias_ = None
        self.classes_ = None
        self.loss_history_ = []

    def _sigmoid(self, z): # Numerically stable sigmoid
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def _fit_lbfgs(self, X, y): # L-BFGS optimization from scratch
        n_samples, n_features = X.shape
        m = 10  # Memory size

        # L-BFGS history
        s_history, y_history, rho_history = [], [], []

        # Initialize parameters
        params = np.concatenate([self.weights_, [self.bias_]])

        def objective_and_gradient(params):
            w, b = params[:-1], params[-1]

            # Forward pass
            z = X @ w + b
            y_pred = self._sigmoid(z)
            y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)

            # Loss (BCE + L2 regularization scaled by n_samples)
            bce = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
            reg_term = (
                (1 / (2 * self.C * n_samples)) * np.sum(w**2)
                if self.regularization == "l2"
                else 0
            )
            loss = bce + reg_term

            # Gradient
            error = y_pred - y
            dw = (X.T @ error) / n_samples
            if self.regularization == "l2":
                dw += (1 / (self.C * n_samples)) * w
            db = np.mean(error)

            return loss, np.concatenate([dw, [db]])

        def lbfgs_two_loop(grad, s_hist, y_hist, rho_hist): # L-BFGS two-loop recursion
            q = grad.copy()
            m_curr = len(s_hist)

            if m_curr == 0:
                return -grad

            alpha = np.zeros(m_curr)
            for i in range(m_curr - 1, -1, -1):
                alpha[i] = rho_hist[i] * np.dot(s_hist[i], q)
                q = q - alpha[i] * y_hist[i]

            gamma = np.dot(s_hist[-1], y_hist[-1]) / (
                np.dot(y_hist[-1], y_hist[-1]) + 1e-10
            )
            r = gamma * q

            for i in range(m_curr):
                beta = rho_hist[i] * np.dot(y_hist[i], r)
                r = r + (alpha[i] - beta) * s_hist[i]

            return -r

        def line_search(params, direction, grad, loss, max_iter=20): # Backtracking line search with Armijo condition
            c1 = 1e-4
            alpha = 1.0
            rho = 0.5
            directional_deriv = np.dot(grad, direction)

            for _ in range(max_iter):
                new_params = params + alpha * direction
                new_loss, _ = objective_and_gradient(new_params)
                if new_loss <= loss + c1 * alpha * directional_deriv:
                    return alpha, new_params, new_loss
                alpha *= rho

            return (
                alpha,
                params + alpha * direction,
                objective_and_gradient(params + alpha * direction)[0],
            )

        # Main L-BFGS loop
        loss, grad = objective_and_gradient(params)
        self.loss_history_ = [loss]

        for iteration in range(self.max_iter):
            direction = lbfgs_two_loop(grad, s_history, y_history, rho_history)
            alpha, new_params, new_loss = line_search(params, direction, grad, loss)
            _, new_grad = objective_and_gradient(new_params)

            s = new_params - params
            y_vec = new_grad - grad

            if np.linalg.norm(new_grad) < self.tol:
                if self.verbose:
                    print(f"L-BFGS converged at iteration {iteration}")
                params = new_params
                break

            sy = np.dot(s, y_vec)
            if sy > 1e-10:
                if len(s_history) >= m:
                    s_history.pop(0)
                    y_history.pop(0)
                    rho_history.pop(0)
                s_history.append(s)
                y_history.append(y_vec)
                rho_history.append(1.0 / sy)

            params = new_params
            loss = new_loss
            grad = new_grad
            self.loss_history_.append(loss)

        self.weights_ = params[:-1]
        self.bias_ = params[-1]

    def fit(self, X, y): # Fit the logistic regression model
        X = np.array(X, dtype=np.float64)
        y = np.array(y, dtype=np.float64)

        self.classes_ = np.unique(y)

        if self.random_state is not None:
            np.random.seed(self.random_state)

        self.weights_ = np.random.randn(X.shape[1]) * 0.01
        self.bias_ = 0.0

        self._fit_lbfgs(X, y)
        return self

    def predict_proba(self, X): # Predict class probabilities
        X = np.array(X, dtype=np.float64)
        z = X @ self.weights_ + self.bias_
        prob_1 = self._sigmoid(z)
        return np.column_stack([1 - prob_1, prob_1])

    def predict(self, X, threshold=0.5): # Predict class labels
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


print("LogisticRegressionScratch class defined!")

In [ ]:
print("=" * 60)
print("Training Logistic Regression (From Scratch)")
print("=" * 60)

# Configuration: C=1.0, L-BFGS, sigmoid cv=4
base_lr = LogisticRegressionScratch(
    max_iter=200, C=1.0, regularization="l2", tol=1e-6, random_state=42
)

# With sigmoid calibration
lr_calibrated = CalibratedClassifierCVScratch(
    base_estimator=base_lr, method="sigmoid", cv=4, random_state=42
)

# Train on full data
lr_calibrated.fit(X_38_scaled, y)
print("Logistic Regression trained!")

# Evaluate on training data
lr_train_proba = lr_calibrated.predict_proba(X_38_scaled)[:, 1]
lr_train_auc = roc_auc_score(y, lr_train_proba)
print(f"LR Training AUC: {lr_train_auc:.5f}")

#### Analysis

In [ ]:
# <<<<<<<<<<<< TOLONG DI CEK LAGI BAGIAN INI NTAR
print("=" * 60)
print("MODEL ANALYSIS")
print("=" * 60)
# 1. Train uncalibrated model untuk ekstraksi weights
print("\n[1] Training uncalibrated LR for weight extraction...")
lr_uncalibrated = LogisticRegressionScratch(
    max_iter=200, C=1.0, regularization="l2", tol=1e-6, random_state=42
)
lr_uncalibrated.fit(X_38_scaled, y)
print(f"Weights shape: {lr_uncalibrated.weights_.shape}")
print(f"Bias: {lr_uncalibrated.bias_:.6f}")
# 2. Feature Weights Analysis
print("\n[2] Feature Weights Analysis")
print("-" * 60)
weight_df = pd.DataFrame({
    "feature": FEATURES_38,
    "weight": lr_uncalibrated.weights_,
    "abs_weight": np.abs(lr_uncalibrated.weights_)
})
weight_df = weight_df.sort_values("abs_weight", ascending=False)
print("\nTop 10 Features (by absolute weight):")
print()
top_10 = weight_df.head(10)
for i, (_, row) in enumerate(top_10.iterrows(), 1):
    sign = "+" if row['weight'] > 0 else ""
    effect = "(increases fraud)" if row['weight'] > 0 else "(decreases fraud)"
    print(f"  {i:2}. {row['feature']:<28} {sign}{row['weight']:.6f}  {effect}")
# 3. Confusion Matrix & Classification Report
print("\n[3] Confusion Matrix & Classification Report")
print("-" * 60)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
# Split data
X_tr, X_val, y_tr, y_val = train_test_split(
    X_38_scaled, y, test_size=0.2, random_state=42, stratify=y
)
# Train dan predict
lr_val = LogisticRegressionScratch(
    max_iter=200, C=1.0, regularization="l2", tol=1e-6, random_state=42
)
lr_val.fit(X_tr, y_tr)
y_proba = lr_val.predict_proba(X_val)[:, 1]
y_pred = lr_val.predict(X_val, threshold=0.5)
# Metrics
val_auc = roc_auc_score(y_val, y_proba)
cm = confusion_matrix(y_val, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"\nValidation AUC-ROC: {val_auc:.5f}")
print(f"\nConfusion Matrix:")
print(f"                 Predicted")
print(f"                 Normal  Fraud")
print(f"  Actual Normal  {tn:>6}  {fp:>6}")
print(f"         Fraud   {fn:>6}  {tp:>6}")
print(f"\nBreakdown:")
print(f"  TN (True Negative):  {tn:,}")
print(f"  FP (False Positive): {fp:,}")
print(f"  FN (False Negative): {fn:,}")
print(f"  TP (True Positive):  {tp:,}")
print(f"\nError Analysis:")
print(f"  Total errors: {fp + fn:,} ({(fp + fn)/len(y_val)*100:.2f}%)")
print(f"  FP rate: {fp/(fp+tn)*100:.2f}% (normal flagged as fraud)")
print(f"  FN rate: {fn/(fn+tp)*100:.2f}% (fraud missed)")
print(f"  Dominant error type: {'False Negative' if fn > fp else 'False Positive'}")
print(f"\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=["Normal", "Fraud"], digits=4))
# 4. Cross-Validation
print("[4] Cross-Validation (5-Fold)")
print("-" * 60)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {"auc": [], "precision": [], "recall": [], "tn": [], "fp": [], "fn": [], "tp": []}
for fold, (train_idx, val_idx) in enumerate(cv.split(X_38_scaled, y), 1):
    X_t, X_v = X_38_scaled[train_idx], X_38_scaled[val_idx]
    y_t, y_v = y[train_idx], y[val_idx]
    
    model = LogisticRegressionScratch(max_iter=200, C=1.0, regularization="l2", random_state=42)
    model.fit(X_t, y_t)
    
    proba = model.predict_proba(X_v)[:, 1]
    pred = model.predict(X_v)
    
    tn_cv, fp_cv, fn_cv, tp_cv = confusion_matrix(y_v, pred).ravel()
    
    cv_results["auc"].append(roc_auc_score(y_v, proba))
    cv_results["precision"].append(tp_cv / (tp_cv + fp_cv) if (tp_cv + fp_cv) > 0 else 0)
    cv_results["recall"].append(tp_cv / (tp_cv + fn_cv) if (tp_cv + fn_cv) > 0 else 0)
    cv_results["tn"].append(tn_cv)
    cv_results["fp"].append(fp_cv)
    cv_results["fn"].append(fn_cv)
    cv_results["tp"].append(tp_cv)
    
    print(f"  Fold {fold}: AUC={cv_results['auc'][-1]:.4f}, Precision={cv_results['precision'][-1]:.4f}, Recall={cv_results['recall'][-1]:.4f}")
print(f"\nCV Summary:")
print(f"  AUC-ROC:    {np.mean(cv_results['auc']):.4f} +/- {np.std(cv_results['auc']):.4f}")
print(f"  Precision:  {np.mean(cv_results['precision']):.4f} +/- {np.std(cv_results['precision']):.4f}")
print(f"  Recall:     {np.mean(cv_results['recall']):.4f} +/- {np.std(cv_results['recall']):.4f}")
print(f"\nTotal CM across all folds:")
print(f"  TN: {sum(cv_results['tn']):,}  FP: {sum(cv_results['fp']):,}")
print(f"  FN: {sum(cv_results['fn']):,}  TP: {sum(cv_results['tp']):,}")
print("\n" + "=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)

In [ ]:
print("=" * 60)
print("Cross-Validation Evaluation")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# LR CV evaluation
lr_cv_scores = []
for train_idx, val_idx in cv.split(X_38_scaled, y):
    X_tr, X_va = X_38_scaled[train_idx], X_38_scaled[val_idx]
    y_tr, y_va = y[train_idx], y[val_idx]

    base_lr_cv = LogisticRegressionScratch(
        max_iter=200, C=1.0, regularization="l2", random_state=42
    )
    lr_cv = CalibratedClassifierCVScratch(
        base_lr_cv, method="sigmoid", cv=4, random_state=42
    )
    lr_cv.fit(X_tr, y_tr)
    proba = lr_cv.predict_proba(X_va)[:, 1]
    lr_cv_scores.append(roc_auc_score(y_va, proba))

print(f"LR CV AUC: {np.mean(lr_cv_scores):.5f} (+/- {np.std(lr_cv_scores):.5f})")

SKLearn

In [ ]:
from sklearn.linear_model import LogisticRegression as SklearnLR

print("\n" + "=" * 60)
print("Logistic Regression: Scratch vs Sklearn")
print("=" * 60)

# Use a subset for faster comparison
X_sample = X_38_scaled[:10000]
y_sample = y[:10000]

# Sklearn LR
sklearn_lr = SklearnLR(C=1.0, solver="lbfgs", max_iter=200, random_state=42)
sklearn_lr.fit(X_sample, y_sample)
sklearn_lr_proba = sklearn_lr.predict_proba(X_sample)[:, 1]

# Scratch LR
scratch_lr = LogisticRegressionScratch(C=1.0, max_iter=200, tol=1e-6, random_state=42)
scratch_lr.fit(X_sample, y_sample)
scratch_lr_proba = scratch_lr.predict_proba(X_sample)[:, 1]

# Compare
lr_correlation = np.corrcoef(sklearn_lr_proba, scratch_lr_proba)[0, 1]
lr_mae = np.mean(np.abs(sklearn_lr_proba - scratch_lr_proba))

print(f"Correlation: {lr_correlation:.6f}")
print(f"Mean Absolute Error: {lr_mae:.6f}")
print(f"Sklearn AUC: {roc_auc_score(y_sample, sklearn_lr_proba):.5f}")
print(f"Scratch AUC: {roc_auc_score(y_sample, scratch_lr_proba):.5f}")
print(f"Match: {'YES' if lr_correlation > 0.999 else 'CLOSE'}")

## C. KNN

In [ ]:
import pickle

class KNN:    
    def __init__(self, k=3, metric='euclidean', weights='uniform'):
        # Inisialisasi KNN
        if k < 1:
            raise ValueError("k must be >= 1")
        if metric not in ['euclidean', 'manhattan']:
            raise ValueError("metric harus 'euclidean' atau 'manhattan'")
        if weights not in ['uniform', 'distance']:
            raise ValueError("weights harus 'uniform' atau 'distance'")
        
        self.k = k
        self.metric = metric
        self.weights = weights
        self.X_train = None
        self.y_train = None
    
    def fit(self, X_train, y_train):
        self.X_train = np.asarray(X_train)
        self.y_train = np.asarray(y_train)
        
        if len(self.X_train) == 0:
            raise ValueError("X_train tidak boleh kosong")
        if len(self.X_train) != len(self.y_train):
            raise ValueError("X_train dan y_train harus memiliki jumlah sampel yang sama")
        
        return self
    
    def _calculate_distance(self, x_query):
        if self.metric == 'euclidean':
            # Euclidean distance: sqrt(sum((x - y)^2))
            differences = self.X_train - x_query
            distances = np.sqrt(np.sum(differences ** 2, axis=1))
        elif self.metric == 'manhattan':
            # Manhattan distance: sum(|x - y|)
            differences = self.X_train - x_query
            distances = np.sum(np.abs(differences), axis=1)
        else:
            raise ValueError(f"Metric tidak diketahui: {self.metric}")
        
        return distances
    
    def _predict_one(self, x_query):
        if self.X_train is None:
            raise RuntimeError("Model must be fitted before prediction")
        
        # Hitung jarak ke semua training points
        distances = self._calculate_distance(x_query)
        
        # Cari k yang terdekat dari neighbors
        k_indices = np.argsort(distances)[:self.k]
        k_nearest_labels = self.y_train[k_indices]
        k_nearest_dists = distances[k_indices]
        
        # Voting 
        if self.weights == 'uniform':
            # Uniform voting
            num_class_1 = np.sum(k_nearest_labels)
            proba = num_class_1 / self.k
            prediction = 1 if proba > 0.5 else 0
            
        else:  # self.weights == 'distance'
            # Distance-weighted voting
            epsilon = 1e-5  # Avoid division by zero
            
            # Bobot = 1 / (distance + epsilon)
            weights_array = 1.0 / (k_nearest_dists + epsilon)
            
            # Hitung weighted sum untuk class 1
            weighted_sum_1 = np.sum(weights_array[k_nearest_labels == 1])
            total_weight = np.sum(weights_array)
            
            # Probability adalah weighted sum / total weight
            proba = weighted_sum_1 / total_weight
            prediction = 1 if proba > 0.5 else 0
        
        return prediction, proba
    
    def predict(self, X_test):
        X_test = np.asarray(X_test)
        predictions = []
        
        for x in X_test:
            pred, _ = self._predict_one(x)
            predictions.append(pred)
        
        return np.array(predictions)
    
    def predict_proba(self, X_test):
        X_test = np.asarray(X_test)
        probabilities = []
        
        for x in X_test:
            _, proba = self._predict_one(x)
            probabilities.append([1 - proba, proba])
        
        return np.array(probabilities)
    
    def save_model(self, filename):
        with open(filename, 'wb') as f:
            pickle.dump(self, f)
        print(f"Model disimpan pada {filename}")
    
    @staticmethod
    def load_model(filename):
        with open(filename, 'rb') as f:
            model = pickle.load(f)
        print(f"Model loaded from {filename}")
        return model


print("KNN class defined successfully!")

In [ ]:
# Split data untuk KNN
X_train_38_scaled, X_val_38_scaled, y_train, y_val = train_test_split(
    X_38_scaled, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set size: {X_train_38_scaled.shape}")
print(f"Validation set size: {X_val_38_scaled.shape}")

In [ ]:
# KNN FROM SCRATCH 
print("=" * 60)
print("KNN FROM SCRATCH")
print("=" * 60)

# Hyperparameter
k_values = [3, 5, 7]
metrics = ['euclidean', 'manhattan']
weights = ['uniform', 'distance']

# Dictionary untuk menyimpan results
knn_models = {}
knn_results = {}
knn_params = {}

total_configs = len(k_values) * len(metrics) * len(weights)
current_config = 0

print(f"\nTraining {total_configs} configurations...")
print(f"This may take a few minutes. Please wait...\n")

for metric in metrics:
    for weight in weights:
        for k in k_values:
            current_config += 1
            model_name = f"knn_k{k}_{metric}_{weight}"
            
            # Progres indikator
            progress = f"[{current_config}/{total_configs}]"
            print(f"{progress} Training {model_name}...", end=" ", flush=True)
            
            try:
                # Create and train KNN
                knn = KNN(k=k, metric=metric, weights=weight)
                knn.fit(X_train_38_scaled, y_train)
                
                # Membuat prediksi
                y_pred_knn = knn.predict(X_val_38_scaled)
                
                # Calculate probabilities 
                y_proba_knn = knn.predict_proba(X_val_38_scaled)[:, 1]
                
                # Calculate metrics
                from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
                
                accuracy = accuracy_score(y_val, y_pred_knn)
                precision = precision_score(y_val, y_pred_knn, zero_division=0)
                recall = recall_score(y_val, y_pred_knn, zero_division=0)
                f1 = f1_score(y_val, y_pred_knn, zero_division=0)
                auc = roc_auc_score(y_val, y_proba_knn)
                
                # Store results
                knn_models[model_name] = knn
                knn_params[model_name] = {'k': k, 'metric': metric, 'weights': weight}
                knn_results[model_name] = {
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1': f1,
                    'auc': auc
                }
                
                print(f"AUC={auc:.5f}")
                
            except Exception as e:
                print(f"ERROR: {str(e)[:60]}")

# KNN terbaik

print("\n" + "=" * 60)
print("KNN terbaik")
print("=" * 60)

if len(knn_results) > 0:
    sorted_results = sorted(knn_results.items(), key=lambda x: x[1]['auc'], reverse=True)
    
    for i, (model_name, metrics_dict) in enumerate(sorted_results[:5], 1):
        print(f"\n#{i} - {model_name}")
        print(f"    Accuracy:  {metrics_dict['accuracy']:.6f}")
        print(f"    Precision: {metrics_dict['precision']:.6f}")
        print(f"    Recall:    {metrics_dict['recall']:.6f}")
        print(f"    F1-Score:  {metrics_dict['f1']:.6f}")
        print(f"    AUC-ROC:   {metrics_dict['auc']:.6f}")
    
    # Model terbaik
    best_model_name = sorted_results[0][0]
    best_knn = knn_models[best_model_name]
    best_metrics = sorted_results[0][1]
    best_params = knn_params[best_model_name]

    print(f"KNN Terbaik dari Scratch: {best_model_name}")
    print(f"Parameters: k={best_params['k']}, metric={best_params['metric']}, weights={best_params['weights']}")
    print(f"AUC-ROC:   {best_metrics['auc']:.6f}")
    print(f"Accuracy:  {best_metrics['accuracy']:.6f}")
    print(f"Precision: {best_metrics['precision']:.6f}")
    print(f"Recall:    {best_metrics['recall']:.6f}")
    print(f"F1-Score:  {best_metrics['f1']:.6f}")
    
    # Simpan best model
    best_knn.save_model("best_knn_scratch_model.pkl")
    print(f"\nBest KNN model saved to: best_knn_scratch_model.pkl")
else:
    print("\nNo results! Check if there were errors during training.")

In [ ]:
# KNN Scikit Learn

from sklearn.neighbors import KNeighborsClassifier

print("=" * 60)
print("KNN WITH SCIKIT-LEARN")
print("=" * 60)

# Dictionary untuk menyimpan results
sklearn_knn_models = {}
sklearn_knn_results = {}
sklearn_knn_params = {}

total_configs = len(k_values) * len(metrics) * len(weights)
current_config = 0

print(f"\nTraining {total_configs} configurations...")
print(f"This may take a few minutes. Please wait...\n")

for metric in metrics:
    for weight in weights:
        for k in k_values:
            current_config += 1
            model_name = f"knn_k{k}_{metric}_{weight}"
            
            # Progress indicator
            progress = f"[{current_config}/{total_configs}]"
            print(f"{progress} Training {model_name}...", end=" ", flush=True)
            
            try:
                # train KNN dengan sklearn
                sklearn_knn = KNeighborsClassifier(
                    n_neighbors=k,
                    metric=metric,
                    weights=weight,
                    algorithm='auto'
                )
                sklearn_knn.fit(X_train_38_scaled, y_train)
                
                # Prediksi
                y_pred_sklearn = sklearn_knn.predict(X_val_38_scaled)
                
                # Probabilitas
                y_proba_sklearn = sklearn_knn.predict_proba(X_val_38_scaled)[:, 1]
                
                # Calculate metrics
                from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
                
                accuracy = accuracy_score(y_val, y_pred_sklearn)
                precision = precision_score(y_val, y_pred_sklearn, zero_division=0)
                recall = recall_score(y_val, y_pred_sklearn, zero_division=0)
                f1 = f1_score(y_val, y_pred_sklearn, zero_division=0)
                auc = roc_auc_score(y_val, y_proba_sklearn)
                
                # hasil
                sklearn_knn_models[model_name] = sklearn_knn
                sklearn_knn_params[model_name] = {'k': k, 'metric': metric, 'weights': weight}
                sklearn_knn_results[model_name] = {
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1': f1,
                    'auc': auc
                }
                
                print(f"AUC={auc:.5f}")
                
            except Exception as e:
                print(f"ERROR: {str(e)[:60]}")

# KNN Scikit learn terbaik

print("\n" + "=" * 60)
print("KNN SCIKIT-LEARN TERBAIK")
print("=" * 60)

if len(sklearn_knn_results) > 0:
    sorted_sklearn_results = sorted(sklearn_knn_results.items(), key=lambda x: x[1]['auc'], reverse=True)
    
    for i, (model_name, metrics_dict) in enumerate(sorted_sklearn_results[:5], 1):
        print(f"\n#{i} - {model_name}")
        print(f"    Accuracy:  {metrics_dict['accuracy']:.6f}")
        print(f"    Precision: {metrics_dict['precision']:.6f}")
        print(f"    Recall:    {metrics_dict['recall']:.6f}")
        print(f"    F1-Score:  {metrics_dict['f1']:.6f}")
        print(f"    AUC-ROC:   {metrics_dict['auc']:.6f}")
    
    # model terbaik
    best_sklearn_model_name = sorted_sklearn_results[0][0]
    best_sklearn_knn = sklearn_knn_models[best_sklearn_model_name]
    best_sklearn_metrics = sorted_sklearn_results[0][1]
    best_sklearn_params = sklearn_knn_params[best_sklearn_model_name]
    
    print(f"KNN Scikit Learn terbaik: {best_sklearn_model_name}")
    print(f"Parameters: k={best_sklearn_params['k']}, metric={best_sklearn_params['metric']}, weights={best_sklearn_params['weights']}")
    print(f"AUC-ROC:   {best_sklearn_metrics['auc']:.6f}")
    print(f"Accuracy:  {best_sklearn_metrics['accuracy']:.6f}")
    print(f"Precision: {best_sklearn_metrics['precision']:.6f}")
    print(f"Recall:    {best_sklearn_metrics['recall']:.6f}")
    print(f"F1-Score:  {best_sklearn_metrics['f1']:.6f}")
    
    # simpan
    import pickle
    with open("best_knn_sklearn_model.pkl", 'wb') as f:
        pickle.dump(best_sklearn_knn, f)
    print(f"\nBest sklearn KNN model saved to: best_knn_sklearn_model.pkl")
else:
    print("\nNo results! Check if there were errors during training.")

In [ ]:
# PERBANDINGAN: KNN FROM SCRATCH VS SCIKIT-LEARN

import pandas as pd

print("=" * 60)
print("PERBANDINGAN: KNN FROM SCRATCH vs SCIKIT-LEARN")
print("=" * 60)

# perbandingan dataframe
comparison_data = []

for model_name in knn_results.keys():
    scratch_metrics = knn_results[model_name]
    sklearn_metrics = sklearn_knn_results[model_name]
    
    auc_diff = abs(scratch_metrics['auc'] - sklearn_metrics['auc'])
    accuracy_diff = abs(scratch_metrics['accuracy'] - sklearn_metrics['accuracy'])
    
    comparison_data.append({
        'Model': model_name,
        'Scratch AUC': scratch_metrics['auc'],
        'Sklearn AUC': sklearn_metrics['auc'],
        'AUC Difference': auc_diff,
        'Scratch Accuracy': scratch_metrics['accuracy'],
        'Sklearn Accuracy': sklearn_metrics['accuracy'],
        'Accuracy Diff': accuracy_diff
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('AUC Difference', ascending=True)

print("\nComparison Table (sorted by AUC difference):\n")
print(comparison_df.to_string(index=False))

# Statistik perbandingan
print("\n" + "=" * 60)
print("STATISTIK PERBANDINGAN")
print("=" * 60)

mean_auc_diff = comparison_df['AUC Difference'].mean()
max_auc_diff = comparison_df['AUC Difference'].max()
min_auc_diff = comparison_df['AUC Difference'].min()

mean_acc_diff = comparison_df['Accuracy Diff'].mean()
max_acc_diff = comparison_df['Accuracy Diff'].max()
min_acc_diff = comparison_df['Accuracy Diff'].min()

print(f"\nAUC Differences:")
print(f"  Mean: {mean_auc_diff:.6f}")
print(f"  Min:  {min_auc_diff:.6f}")
print(f"  Max:  {max_auc_diff:.6f}")

print(f"\nAccuracy Differences:")
print(f"  Mean: {mean_acc_diff:.6f}")
print(f"  Min:  {min_acc_diff:.6f}")
print(f"  Max:  {max_acc_diff:.6f}")

# model terbaik
best_scratch = max(knn_results.items(), key=lambda x: x[1]['auc'])
best_sklearn = max(sklearn_knn_results.items(), key=lambda x: x[1]['auc'])

print(f"\nBEST FROM SCRATCH: {best_scratch[0]}")
print(f"   AUC-ROC:  {best_scratch[1]['auc']:.6f}")
print(f"   Accuracy: {best_scratch[1]['accuracy']:.6f}")

print(f"\nBEST SKLEARN: {best_sklearn[0]}")
print(f"   AUC-ROC:  {best_sklearn[1]['auc']:.6f}")
print(f"   Accuracy: {best_sklearn[1]['accuracy']:.6f}")

auc_final_diff = abs(best_scratch[1]['auc'] - best_sklearn[1]['auc'])
print(f"\nFinal AUC Difference: {auc_final_diff:.6f}")

if auc_final_diff < 0.01:
    print("Implementations are nearly identical (diff < 0.01)")
elif auc_final_diff < 0.05:
    print("Implementations are very similar (diff < 0.05)")
else:
    print("Implementations have noticeable differences (diff >= 0.05)")

In [ ]:
# VISUALISASI: KNN FROM SCRATCH vs SCIKIT-LEARN

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Perbandingan KNN: From Scratch vs Scikit-Learn', fontsize=16, fontweight='bold')

# Ekstraksi data
model_names = list(knn_results.keys())
scratch_aucs = [knn_results[m]['auc'] for m in model_names]
sklearn_aucs = [sklearn_knn_results[m]['auc'] for m in model_names]
differences = [abs(scratch_aucs[i] - sklearn_aucs[i]) for i in range(len(model_names))]

# 1. AUC Comparison (Bar chart)
ax1 = axes[0, 0]
x = np.arange(len(model_names[:6])) 
width = 0.35

bars1 = ax1.bar(x - width/2, scratch_aucs[:6], width, label='From Scratch', alpha=0.8)
bars2 = ax1.bar(x + width/2, sklearn_aucs[:6], width, label='Scikit-Learn', alpha=0.8)

ax1.set_xlabel('Model')
ax1.set_ylabel('AUC-ROC')
ax1.set_title('AUC-ROC Comparison (Top 6 Models)')
ax1.set_xticks(x)
ax1.set_xticklabels([n.replace('knn_', '').replace('_', '\n') for n in model_names[:6]], fontsize=8)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2. Scatter plot: Scratch vs Sklearn
ax2 = axes[0, 1]
ax2.scatter(scratch_aucs, sklearn_aucs, s=100, alpha=0.6, edgecolors='black')
min_val = min(min(scratch_aucs), min(sklearn_aucs))
max_val = max(max(scratch_aucs), max(sklearn_aucs))
ax2.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect Match', linewidth=2)

for i, name in enumerate(model_names):
    if i % 3 == 0:  
        ax2.annotate(name.replace('knn_', ''), (scratch_aucs[i], sklearn_aucs[i]), 
                    fontsize=7, alpha=0.7)

ax2.set_xlabel('AUC-ROC (From Scratch)')
ax2.set_ylabel('AUC-ROC (Scikit-Learn)')
ax2.set_title('AUC-ROC Scatter Comparison')
ax2.legend()
ax2.grid(alpha=0.3)

# 3. Difference distribution
ax3 = axes[1, 0]
ax3.bar(range(len(differences)), differences, color='steelblue', alpha=0.7)
ax3.axhline(y=np.mean(differences), color='r', linestyle='--', label=f'Mean: {np.mean(differences):.6f}')
ax3.set_xlabel('Model Index')
ax3.set_ylabel('Absolute AUC Difference')
ax3.set_title('AUC Difference Distribution')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# 4. Metrics comparison for best models
ax4 = axes[1, 1]
best_scratch_model = max(knn_results.items(), key=lambda x: x[1]['auc'])[0]
best_sklearn_model = max(sklearn_knn_results.items(), key=lambda x: x[1]['auc'])[0]

metrics_names = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1-Score']
scratch_metrics_vals = [
    knn_results[best_scratch_model]['auc'],
    knn_results[best_scratch_model]['accuracy'],
    knn_results[best_scratch_model]['precision'],
    knn_results[best_scratch_model]['recall'],
    knn_results[best_scratch_model]['f1']
]
sklearn_metrics_vals = [
    sklearn_knn_results[best_sklearn_model]['auc'],
    sklearn_knn_results[best_sklearn_model]['accuracy'],
    sklearn_knn_results[best_sklearn_model]['precision'],
    sklearn_knn_results[best_sklearn_model]['recall'],
    sklearn_knn_results[best_sklearn_model]['f1']
]

x_metrics = np.arange(len(metrics_names))
width = 0.35

ax4.bar(x_metrics - width/2, scratch_metrics_vals, width, label=f'Scratch: {best_scratch_model}', alpha=0.8)
ax4.bar(x_metrics + width/2, sklearn_metrics_vals, width, label=f'Sklearn: {best_sklearn_model}', alpha=0.8)

ax4.set_xlabel('Metrics')
ax4.set_ylabel('Score')
ax4.set_title('Best Models Metrics Comparison')
ax4.set_xticks(x_metrics)
ax4.set_xticklabels(metrics_names, fontsize=9)
ax4.legend(fontsize=8)
ax4.set_ylim([0, 1])
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('knn_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'knn_comparison.png'")

In [ ]:
# BONUS: KNN PROCESS VISUALIZATION 

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("=" * 60)
print("KNN TRAINING PROCESS VISUALIZATION")
print("=" * 60)

print("\nApplying PCA to reduce dimensions to 2D for visualization...")

# PCA to training data
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_38_scaled)
X_val_pca = pca.transform(X_val_38_scaled)

print(f"PCA completed")
print(f"   Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"   Total variance explained: {sum(pca.explained_variance_ratio_):.4f}")

# visualisasi
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('KNN Training Process Visualization (PCA 2D)', fontsize=16, fontweight='bold')

colors = ['blue', 'red']
labels = ['Class 0', 'Class 1']

# Plot 1: Training data distribution
ax = axes[0, 0]
for i, color, label in zip([0, 1], colors, labels):
    mask = y_train == i
    ax.scatter(X_train_pca[mask, 0], X_train_pca[mask, 1], 
              c=color, label=label, alpha=0.6, s=30, edgecolors='black', linewidth=0.5)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
ax.set_title('Training Data Distribution')
ax.legend()
ax.grid(alpha=0.3)

# Plot 2: Validation data distribution
ax = axes[0, 1]
for i, color, label in zip([0, 1], colors, labels):
    mask = y_val == i
    ax.scatter(X_val_pca[mask, 0], X_val_pca[mask, 1], 
              c=color, label=label, alpha=0.6, s=30, edgecolors='black', linewidth=0.5)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
ax.set_title('Validation Data Distribution')
ax.legend()
ax.grid(alpha=0.3)

# Plot 3: Decision boundaries for best model
ax = axes[1, 0]

# KNN model terbaik
best_model_name = max(knn_results.items(), key=lambda x: x[1]['auc'])[0]
best_model = knn_models[best_model_name]

h = 0.02  
x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1

xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

ax.scatter(X_train_pca[y_train == 0, 0], X_train_pca[y_train == 0, 1], 
          c='blue', label='Class 0 (Training)', alpha=0.4, s=20)
ax.scatter(X_train_pca[y_train == 1, 0], X_train_pca[y_train == 1, 1], 
          c='red', label='Class 1 (Training)', alpha=0.4, s=20)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
ax.set_title(f'Training Data with Best Model: {best_model_name}')
ax.legend()
ax.grid(alpha=0.3)

# Plot 4: Model performance statistics
ax = axes[1, 1]
ax.axis('off')

# kesimpulan
summary_text = f"""
KNN TRAINING SUMMARY

Best Model: {best_model_name}
Parameters:
  • k = {best_knn.k}
  • metric = {best_knn.metric}
  • weights = {best_knn.weights}

Performance Metrics:
  • AUC-ROC:   {best_metrics['auc']:.6f}
  • Accuracy:  {best_metrics['accuracy']:.6f}
  • Precision: {best_metrics['precision']:.6f}
  • Recall:    {best_metrics['recall']:.6f}
  • F1-Score:  {best_metrics['f1']:.6f}

Total Models Trained: {len(knn_models)}
Training Data Size: {len(X_train_38_scaled):,}
Validation Data Size: {len(X_val_38_scaled):,}

PCA Information:
  • Explained Variance: {sum(pca.explained_variance_ratio_):.4f}
  • PC1: {pca.explained_variance_ratio_[0]:.4f}
  • PC2: {pca.explained_variance_ratio_[1]:.4f}
"""

ax.text(0.1, 0.9, summary_text, transform=ax.transAxes, fontsize=10,
       verticalalignment='top', fontfamily='monospace',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('knn_training_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTraining visualization saved as 'knn_training_visualization.png'")

## D. Improvements (Optional)

- **Visualize the model evaluation result**

This will help you to understand the details more clearly about your model's performance. From the visualization, you can see clearly if your model is leaning towards a class than the others. (Hint: confusion matrix, ROC-AUC curve, etc.)

- **Explore the hyperparameters of your models**

Each models have their own hyperparameters. And each of the hyperparameter have different effects on the model behaviour. You can optimize the model performance by finding the good set of hyperparameters through a process called **hyperparameter tuning**. (Hint: Grid search, random search, bayesian optimization)

- **Cross-validation**

Cross-validation is a critical technique in machine learning and data science for evaluating and validating the performance of predictive models. It provides a more **robust** and **reliable** evaluation method compared to a hold-out (single train-test set) validation. Though, it requires more time and computing power because of how cross-validation works. (Hint: k-fold cross-validation, stratified k-fold cross-validation, etc.)

In [ ]:
# Hyperparameter Tuning and Visualization for KNN
print("\n" + "=" * 60)
print("KNN Hyperparameter Analysis")
print("=" * 60)

# Create a summary dataframe
knn_summary = []
for model_name, metrics in knn_results.items():
    parts = model_name.split('_')
    k = int(parts[1][1:])
    metric = parts[2]
    weight = parts[3]
    
    knn_summary.append({
        'k': k,
        'metric': metric,
        'weight': weight,
        'accuracy': metrics['accuracy'],
        'precision': metrics['precision'],
        'recall': metrics['recall'],
        'f1': metrics['f1'],
        'auc': metrics['auc']
    })

knn_summary_df = pd.DataFrame(knn_summary)

print("\nKNN Results Summary:")
print(knn_summary_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# AUC by k value
ax1 = axes[0, 0]
for metric in ['euclidean', 'manhattan']:
    for weight in ['uniform', 'distance']:
        data = knn_summary_df[(knn_summary_df['metric'] == metric) & (knn_summary_df['weight'] == weight)]
        if len(data) > 0:
            ax1.plot(data['k'].values, data['auc'].values, marker='o', label=f'{metric}-{weight}')
ax1.set_xlabel('k')
ax1.set_ylabel('AUC')
ax1.set_title('AUC vs k value')
ax1.legend()
ax1.grid(True, alpha=0.3)

# F1-Score comparison
ax2 = axes[0, 1]
knn_summary_df_sorted = knn_summary_df.sort_values('k')
ax2.scatter(knn_summary_df_sorted['k'], knn_summary_df_sorted['f1'], alpha=0.6)
ax2.set_xlabel('k')
ax2.set_ylabel('F1-Score')
ax2.set_title('F1-Score by k value')
ax2.grid(True, alpha=0.3)

# Metric comparison
ax3 = axes[1, 0]
metric_comparison = knn_summary_df.groupby('metric')[['accuracy', 'precision', 'recall', 'f1', 'auc']].mean()
metric_comparison.plot(kind='bar', ax=ax3)
ax3.set_title('Metrics by Distance Metric')
ax3.set_ylabel('Score')
ax3.tick_params(axis='x', rotation=45)

# Weight comparison
ax4 = axes[1, 1]
weight_comparison = knn_summary_df.groupby('weight')[['accuracy', 'precision', 'recall', 'f1', 'auc']].mean()
weight_comparison.plot(kind='bar', ax=ax4)
ax4.set_title('Metrics by Weight Type')
ax4.set_ylabel('Score')
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("Analysis Complete")
print("=" * 60)

## E. Submission
To predict the test set target feature and submit the results to the kaggle competition platform, do the following:
1. Create a new pipeline instance identical to the first in Data Preprocessing
2. With the pipeline, apply `fit_transform` to the original training set before splitting, then only apply `transform` to the test set.
3. Retrain the model on the preprocessed training set
4. Predict the test set
5. Make sure the submission contains the `id` and `attack_cat` column.

In [ ]:
print("=" * 60)
print("Generating Kaggle Submissions")
print("=" * 60)

# Generate predictions on test set for Logistic Regression
lr_test_proba = lr_calibrated.predict_proba(X_test_38_scaled)[:, 1]
print(f"\nLogistic Regression:")
print(f"  Test proba range: [{lr_test_proba.min():.4f}, {lr_test_proba.max():.4f}]")

# Generate predictions on test set for KNN
print(f"\nK-Nearest Neighbors (best model: {best_model_name}):")
print(f"  Generating predictions for {len(test_df)} test samples...")

knn_test_proba = []
for x in X_test_38_scaled:
    distances = np.sqrt(np.sum((best_knn.X_train - x)**2, axis=1))
    k_indices = np.argsort(distances)[:best_knn.k]
    k_nearest_labels = best_knn.y_train[k_indices]
    k_nearest_dists = distances[k_indices]
    
    if best_knn.weights == 'uniform':
        prob_1 = np.sum(k_nearest_labels) / best_knn.k
    else:  # distance weights
        epsilon = 1e-5
        weights_dist = 1 / (k_nearest_dists + epsilon)
        weighted_sum_1 = np.sum(weights_dist[k_nearest_labels == 1])
        prob_1 = weighted_sum_1 / np.sum(weights_dist)
    
    knn_test_proba.append(prob_1)

knn_test_proba = np.array(knn_test_proba)
print(f"  Test proba range: [{knn_test_proba.min():.4f}, {knn_test_proba.max():.4f}]")

# Create submission dataframes
submission_lr = pd.DataFrame({"ID": test_ids, "is_fraud": lr_test_proba})
submission_knn = pd.DataFrame({"ID": test_ids, "is_fraud": knn_test_proba})

# Create ensemble submission (average of both models)
submission_ensemble = pd.DataFrame({"ID": test_ids, "is_fraud": (lr_test_proba + knn_test_proba) / 2})

# Save submissions
submission_lr.to_csv("submission_lr.csv", index=False)
submission_knn.to_csv("submission_knn.csv", index=False)
submission_ensemble.to_csv("submission_ensemble.csv", index=False)

print("\nSubmissions saved:")
print("  - submission_lr.csv")
print(f"  - submission_knn.csv (best KNN: {best_model_name})")
print("  - submission_ensemble.csv (average of LR and KNN)")

In [ ]:
import base64
from IPython.display import HTML

submission_filename = "submission_lr.csv"

with open(submission_filename, "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

payload = f'<a download="{submission_filename}" href="data:text/csv;base64,{b64}" target="_blank">Click here to Download {submission_filename}</a>'
HTML(payload)

# 6. Error Analysis

Based on all the process you have done until the modeling and evaluation step, write an analysis to support each steps you have taken to solve this problem. Write the analysis using the markdown block. Some questions that may help you in writing the analysis:

- Does my model perform better in predicting one class than the other? If so, why is that?
- To each models I have tried, which performs the best and what could be the reason?
- Is it better for me to impute or drop the missing data? Why?
- Does feature scaling help improve my model performance?
- etc...

## Summary of Results

| Model | Local CV AUC | Kaggle Score | Implementation |
|-------|--------------|--------------|----------------|
| **Logistic Regression** | ~0.607 | **0.62783** | From scratch (L-BFGS) |
| **K-Nearest Neighbors** | TBD | TBD | From scratch with uniform & distance weighting |
| **Ensemble (LR + KNN)** | TBD | TBD | Average of LR and KNN probabilities |